# 04 — Kiểm chứng CLI/runner PDP ResNet và ConvNeXt-V2-Base

Chạy đường training thật qua `scripts/run_exp.sh` và `pdp/main.py`: train → dừng ở optimizer step 1 → tiến trình mới resume tới step 4 → predict-only → skip task đã xong. Hai backbone, mỗi backbone ở 640/800; batch 1, effective batch 2, FP32, seed 0. Không patch factory/model/loader.

Phụ thuộc notebook 02 và 03 đã PASS, Conda `pdp` Python 3.10 và CUDA. **Restart kernel trước khi chạy.** Không cài package. Mặc định dùng weights đã cache; có thể đặt `offline=False` để tải public theo revision khóa.

Chỉ dùng 4 ảnh train Task 1 và 2 ảnh mỗi tập val/test trong bản JSON nhỏ dưới artifact; không sửa release/holdout. Predictions trên subset chỉ kiểm chứng kỹ thuật, không dùng làm metrics nghiên cứu. PASS là `PASS_PDP_CLI_RUNNER_CUDA_SMOKE`; convergence, full teacher Task 2+, native kernel, calibration và Vast.ai vẫn là gate riêng.


## 1. Cấu hình và bằng chứng run

Giữ mặc định để nghiệm thu đủ bốn trường hợp. Roots nhận `AUTOCHECKOUT_DATA_ROOT` / `AUTOCHECKOUT_RELEASE_ROOT`; summary có thể đặt qua `AUTOCHECKOUT_BASELINE_SUMMARY` / `AUTOCHECKOUT_CONVNEXT_SUMMARY` khi chuyển máy. Artifacts và checkpoints lớn nằm trong `runs/`, không vào git.


In [ ]:
import os, sys, json, uuid, shutil, subprocess, time
from pathlib import Path
from importlib import metadata

REPO = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdp/main.py").exists()
)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from autocheckout.io import load_json, save_json, md5_file
from autocheckout.model_acceptance import RunEvidence
from tools.download_pdp_pretrained import (
    download_pretrained,
    DEFAULT_REVISION,
    sha256_file,
)

RAW = Path(os.getenv("AUTOCHECKOUT_DATA_ROOT", str(REPO / "data/archive"))).resolve()
RELEASE = Path(
    os.getenv(
        "AUTOCHECKOUT_RELEASE_ROOT",
        str(REPO / "data/processed/rpc_100-4x25_seed0_native_v1"),
    )
).resolve()
CONFIG = {
    "seed": 0,
    "resolutions": [640, 800],
    "experiments": ["EXP-B1", "EXP-B2"],
    "batch_size": 1,
    "effective_batch": 2,
    "train_samples": 4,
    "prediction_samples": 2,
    "offline": True,
    "raw_root": str(RAW),
    "release_root": str(RELEASE),
    "backbone_revision": "9e250ed8f88b436472d2b24af26f82a8aa8c719d",
}
ARTIFACT = REPO / "runs/modeling/training_runtime" / str(uuid.uuid4())
CASES = [
    (name, resolution)
    for name in CONFIG["experiments"]
    for resolution in CONFIG["resolutions"]
]
REQUIRED = [
    "environment",
    "prior_acceptance",
    "release_lock",
    "weights",
    "regression",
    *[f"{name}_{resolution}" for name, resolution in CASES],
    "sources_preserved",
]
RUN = RunEvidence(
    ARTIFACT, CONFIG, REQUIRED, pass_status="PASS_PDP_CLI_RUNNER_CUDA_SMOKE"
)
RUN.data.update(
    git_commit=subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
    ).strip(),
    full_training_ready=False,
    native_kernel_accepted=False,
    python=sys.version,
    executable=sys.executable,
)
SOURCE_BEFORE = {}
for relative in [
    "pdp/main.py",
    "pdp/engine.py",
    "pdp/runtime.py",
    "pdp/checkpointing.py",
    "pdp/inference.py",
    "pdp/models/backbones.py",
    "pdp/models/modeling_deformable_detr.py",
    "autocheckout/model_state.py",
    "tools/pdp_runtime_acceptance.py",
    "scripts/run_exp.sh",
    "requirements.txt",
    "configs/exp/native/common.sh",
    "configs/exp/native/EXP-B1.sh",
    "configs/exp/native/EXP-B2.sh",
]:
    path = REPO / relative
    SOURCE_BEFORE[str(path)] = ("sha256", sha256_file(path))
    target = ARTIFACT / "source" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, target)
shutil.copy2(
    REPO / "notebooks/modeling/04_training_runtime_acceptance.ipynb",
    ARTIFACT / "notebook_source.ipynb",
)
RUN.persist()
print("Run", ARTIFACT.name, "started", RUN.data["started_at"], "outputs", ARTIFACT)
ENV = {
    **os.environ,
    "USE_TF": "0",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "PYTHONPATH": str(REPO),
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
}


def command_log(command, path, timeout=1800):
    start = time.monotonic()
    with path.open("w") as log:
        process = subprocess.Popen(
            command, cwd=REPO, env=ENV, stdout=log, stderr=subprocess.STDOUT
        )
        try:
            while process.poll() is None:
                if time.monotonic() - start > timeout:
                    raise TimeoutError(str(path))
                print(path.name, round(time.monotonic() - start), "s", flush=True)
                time.sleep(10)
            return process.returncode
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait(timeout=30)

Run ac818116-ceb0-4ce8-9c1e-979c6f54788a started 2026-10-09T23:58:00+07:00 outputs /home/sakana/Code/AutoCheckout-CL/runs/modeling/training_runtime/ac818116-ceb0-4ce8-9c1e-979c6f54788a


## 2. Environment và các nghiệm thu trước

Đọc summary thật, đủ gates; không dùng dòng PASS trong docs thay cho artifacts. Runtime subprocess dùng Python của kernel hiện tại. Native CUDA kernel không được nghiệm thu ở đây; kết quả fallback được ghi riêng.


In [ ]:
with RUN.gate("environment") as evidence:
    assert sys.version_info[:2] == (3, 10), "Select Conda pdp Python 3.10"
    pins = {
        "torch": "2.2.2",
        "torchvision": "0.17.2",
        "timm": "0.9.12",
        "transformers": "4.37.2",
        "pytorch-lightning": "2.1.3",
        "packaging": "24.2",
        "typing-extensions": "4.13.2",
    }
    versions = {name: metadata.version(name) for name in pins}
    assert all(versions[k].split("+")[0] == v for k, v in pins.items()), versions
    check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], capture_output=True, text=True
    )
    (ARTIFACT / "pip_check.log").write_text(check.stdout + check.stderr)
    assert check.returncode == 0, check.stdout + check.stderr
    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))",
        ],
        env=ENV,
        capture_output=True,
        text=True,
    )
    assert probe.returncode == 0, probe.stderr
    evidence.update(packages=versions, gpu=probe.stdout.strip(), pip_check_returncode=0)
with RUN.gate("prior_acceptance") as evidence:
    inputs = [
        (
            "full_pdp_baseline",
            "AUTOCHECKOUT_BASELINE_SUMMARY",
            "PASS_FULL_PDP_CUDA_SMOKE",
            [
                "environment",
                "foundation",
                "release_lock",
                "pretrained_files",
                "cuda_640",
                "cuda_800",
                "sources_preserved",
            ],
        ),
        (
            "convnext_adapter",
            "AUTOCHECKOUT_CONVNEXT_SUMMARY",
            "PASS_CONVNEXT_ADAPTER_CUDA_SMOKE",
            [
                "environment",
                "baseline_evidence",
                "release_lock",
                "weights",
                "adapter_regression",
                "cuda_640",
                "cuda_800",
                "sources_preserved",
            ],
        ),
    ]
    for folder, variable, status, gates in inputs:
        root = REPO / "runs/modeling" / folder
        path = (
            Path(os.environ[variable]).expanduser()
            if os.getenv(variable)
            else root / load_json(root / "latest.json")["run_id"] / "summary.json"
        )
        summary = load_json(path)
        assert summary["status"] == status and all(
            summary["gates"].get(n, {}).get("status") == "PASS" for n in gates
        )
        evidence[folder] = {
            "run_id": summary["run_id"],
            "summary_sha256": sha256_file(path),
        }
        shutil.copy2(path, ARTIFACT / f"{folder}_summary.json")

## 3. Release khóa và subset kiểm chứng

Hash tất cả split/task JSON, mapping và raw annotation nguồn. Tạo bản subset dưới artifact với cùng bbox native. Tập train giữ current-task labels; không thêm full GT vào train. Tập val/test chỉ inference và kiểm chứng tọa độ; các chỉ số nhỏ này không phải đánh giá chất lượng.


In [ ]:
with RUN.gate("release_lock") as evidence:
    from autocheckout.taskcfg import TaskConfig

    lock = load_json(REPO / "configs/data/rpc_100-4x25_seed0_native_v1.json")
    tasks = RELEASE / "tasks/real"
    task_config = RELEASE / "task_config.json"
    mapping = TaskConfig.load(task_config)
    assert [t.num_classes for t in mapping.data_tasks] == [100, 25, 25, 25, 25]
    assert (mapping.num_slots, mapping.num_classes, mapping.seed) == (224, 225, 0)
    assert lock["coordinates"] == "native_pixels" and not lock["synthetic_included"]
    for path, digest in [
        (task_config, lock["input"]["task_config_sha256"]),
        (tasks / "manifest.json", lock["task_manifest_sha256"]),
    ]:
        assert sha256_file(path) == digest
        SOURCE_BEFORE[str(path)] = ("sha256", digest)
    for name, digest in lock["input"]["source_sha256"].items():
        path = RAW / name
        assert sha256_file(path) == digest
        SOURCE_BEFORE[str(path)] = ("sha256", digest)
    for name, digest in lock["split_md5"].items():
        path = RELEASE / "splits" / name
        assert md5_file(path) == digest
        SOURCE_BEFORE[str(path)] = ("md5", digest)
    for name, row in load_json(tasks / "manifest.json")["files"].items():
        path = tasks / name
        assert md5_file(path) == row["md5"]
        SOURCE_BEFORE[str(path)] = ("md5", row["md5"])
    SUBSET = ARTIFACT / "subset"
    SUBSET.mkdir()
    selections = {}
    for name, count in [
        ("train_task_1.json", 4),
        ("val_task_1.json", 2),
        ("val_full.json", 2),
        ("test_full.json", 2),
    ]:
        original = load_json(tasks / name)
        images = sorted(original["images"], key=lambda row: row["id"])[:count]
        assert len(images) == count
        ids = {row["id"] for row in images}
        subset = {
            **original,
            "images": images,
            "annotations": [a for a in original["annotations"] if a["image_id"] in ids],
        }
        if name == "train_task_1.json":
            assert all(0 <= a["category_id"] < 100 for a in subset["annotations"])
        save_json(SUBSET / name, subset)
        selections[name] = sorted(ids)
        for row in images:
            path = (RAW / row["file_name"]).resolve()
            assert path.is_relative_to(RAW) and path.is_file()
            SOURCE_BEFORE[str(path)] = ("sha256", sha256_file(path))
    evidence.update(
        protocol=[100, 25, 25, 25, 25], selected_ids=selections, subset_dir=str(SUBSET)
    )
    CONFIG.update(subset_dir=str(SUBSET), task_config=str(task_config))

## 4. Weights và regression

Giữ hai revision của notebook 03. Chạy tests CLI CPU, adapter, paths/teacher fixture, resume, inference, parameter groups và shell runner; GPU fixture phải chạy được, không được skip rồi báo PASS.


In [ ]:
with RUN.gate("weights") as evidence:
    from huggingface_hub import snapshot_download

    CACHE = REPO / "runs/pretrained/hf-cache"
    detector = download_pretrained(
        CACHE, revision=DEFAULT_REVISION, offline=CONFIG["offline"]
    )
    backbone = Path(
        snapshot_download(
            repo_id="timm/convnextv2_base.fcmae_ft_in22k_in1k",
            revision=CONFIG["backbone_revision"],
            allow_patterns=["config.json", "model.safetensors"],
            cache_dir=str(CACHE),
            local_files_only=CONFIG["offline"],
            token=False,
        )
    ).resolve()
    assert backbone.name == CONFIG["backbone_revision"]
    CONFIG.update(
        detector_dir=detector["snapshot_dir"],
        backbone_file=str(backbone / "model.safetensors"),
    )
    files = {}
    for path in [
        Path(CONFIG["detector_dir"]) / name
        for name in ["config.json", "preprocessor_config.json", "model.safetensors"]
    ] + [backbone / "config.json", backbone / "model.safetensors"]:
        digest = sha256_file(path)
        SOURCE_BEFORE[str(path)] = ("sha256", digest)
        files[str(path)] = {"sha256": digest, "bytes": path.stat().st_size}
    evidence.update(files=files, detector_receipt=detector)
    save_json(ARTIFACT / "pretrained_files.json", evidence, indent=2)
    save_json(ARTIFACT / "config.json", CONFIG, indent=2)
with RUN.gate("regression") as evidence:
    import xml.etree.ElementTree as ET

    tests = [
        "test_pdp_runtime.py",
        "test_native_exp_configs.py",
        "test_backbone_adapter.py",
        "test_model_acceptance.py",
        "test_pdp_f8_paths.py",
        "test_pdp_r1_resume.py",
        "test_pdp_f9_inference.py",
        "test_pdp_f10_parameters.py",
        "test_pdp_f13_prior_init.py",
        "test_full_baseline_acceptance.py",
        "test_pdp_b1_finetune.py",
        "test_run_exp.py",
        "test_exp_configs.py",
    ]
    junit = ARTIFACT / "regression.xml"
    command = [
        sys.executable,
        "-m",
        "pytest",
        *[f"tests/{name}" for name in tests],
        "-q",
        "-o",
        "addopts=",
        "--tb=short",
        "--junitxml",
        str(junit),
        "--basetemp",
        str(ARTIFACT / "regression_tmp"),
    ]
    result = command_log(command, ARTIFACT / "regression.log")
    assert result == 0, "See regression.log"
    suites = list(ET.parse(junit).getroot().iter("testsuite"))
    counts = {
        key: sum(int(s.get(key, 0)) for s in suites)
        for key in ["tests", "errors", "failures", "skipped"]
    }
    assert (
        counts["tests"] > 0
        and counts["errors"] == counts["failures"] == counts["skipped"] == 0
    ), counts
    evidence.update(command=command, exit_code=result, counts=counts)
    print(counts)

regression.log 0 s


regression.log 10 s


regression.log 20 s


regression.log 30 s


regression.log 40 s


regression.log 50 s


regression.log 60 s


{'tests': 78, 'errors': 0, 'failures': 0, 'skipped': 0}


## 5. Chạy shell runner thật: 4 trường hợp tuần tự

Mỗi case dùng process riêng. Lần đầu dừng có chủ đích với exit 75 sau step 1; lần thứ hai trỏ weights nguồn tới đường dẫn không tồn tại để chứng minh checkpoint tự đủ config/state, resume tới step 4. Sau đó bỏ một file prediction của chính run mới để kiểm tra runner gọi predict-only, rồi kiểm tra skip task hoàn tất. Cuối cùng dựng model từ final checkpoint để đối chiếu bbox export với bbox chuẩn hóa nhân kích thước ảnh native.

Nếu lỗi, mở `worker.log`, `01_interrupted.log`, `02_resumed.log` hoặc `03_predict_only.log` trong case; không đổi run FAIL thành PASS. Giữ nguyên dữ liệu/checkpoints nguồn.


In [ ]:
for experiment, resolution in CASES:
    name = f"{experiment}_{resolution}"
    output = ARTIFACT / name
    output.mkdir()
    case = {**CONFIG, "experiment": experiment, "resolution": resolution}
    save_json(output / "case.json", case, indent=2)
    with RUN.gate(name) as evidence:
        command = [
            sys.executable,
            "-m",
            "tools.pdp_runtime_acceptance",
            "--config",
            str(output / "case.json"),
            "--output",
            str(output),
        ]
        result = command_log(command, output / "worker.log")
        worker = load_json(output / "result.json")
        assert result == 0 and worker["status"] == "PASS", worker
        evidence.update(worker, command=command)
        print(
            name,
            "PASS",
            worker["peak_training_gpu_gib"],
            "GiB; optimizer step 1 ->",
            worker["optimizer_steps"],
        )

worker.log 0 s


worker.log 10 s


worker.log 20 s


worker.log 30 s


EXP-B1_640 PASS 3.61 GiB; optimizer step 1 -> 4
worker.log 0 s


...

worker.log 10 s


worker.log 20 s


worker.log 30 s


worker.log 40 s


EXP-B2_800 PASS 4.52 GiB; optimizer step 1 -> 4


## 6. Kết luận và lưu outputs

PASS cần đủ 10 gates. Đây là nghiệm thu CLI/runner trên Task 1 subset, không chứng nhận hội tụ hay teacher full ở Task 2. Native kernel chưa có equivalence gate ở notebook này; chưa mở gate calibration/Vast.ai.

Lưu notebook có outputs sau khi hoàn tất, có thể Save As vào `runs/notebook_versions/`. Bước kế tiếp là evaluator/calibration + môi trường training, rồi pilot EXP-B1/EXP-B2 cùng budget.


In [ ]:
with RUN.gate("sources_preserved") as evidence:
    for filename, (algorithm, digest) in SOURCE_BEFORE.items():
        actual = md5_file(filename) if algorithm == "md5" else sha256_file(filename)
        assert actual == digest, f"Source changed: {filename}"
    evidence.update(
        checked_files=len(SOURCE_BEFORE), source_release_holdout_weights_unchanged=True
    )
RUN.finish()
RUN.data.update(
    cli_runner_accepted=RUN.data["status"] == "PASS_PDP_CLI_RUNNER_CUDA_SMOKE",
    full_training_ready=False,
    native_kernel_accepted=False,
    limitations=[
        "Task 1 subset only; no convergence or research metrics.",
        "Full teacher/PPG Task 2+, calibration and Vast.ai remain separate.",
        "Resume state verified with shuffle=0; no claim of identical shuffled data order.",
    ],
)
RUN.persist()
save_json(ARTIFACT / "summary.json", RUN.data, indent=2)
save_json(
    ARTIFACT.parent / "latest.json",
    {
        "run_id": ARTIFACT.name,
        "status": RUN.data["status"],
        "summary": str(ARTIFACT / "summary.json"),
        "finished_at": RUN.data["finished_at"],
    },
    indent=2,
)
print(json.dumps(RUN.data, ensure_ascii=False, indent=2))

{
  "run_id": "ac818116-ceb0-4ce8-9c1e-979c6f54788a",
  "started_at": "2026-10-09T23:58:00+07:00",
  "finished_at": "2026-10-10T00:02:03+07:00",
  "timezone": "Asia/Bangkok",
  "status": "PASS_PDP_CLI_RUNNER_CUDA_SMOKE",
  "config": {
    "seed": 0,
    "resolutions": [
      640,
      800
    ],
    "experiments": [
      "EXP-B1",
      "EXP-B2"
    ],
    "batch_size": 1,
    "effective_batch": 2,
    "train_samples": 4,
    "prediction_samples": 2,
    "offline": true,
    "raw_root": "/home/sakana/Code/AutoCheckout-CL/data/archive",
    "release_root": "/home/sakana/Code/AutoCheckout-CL/data/processed/rpc_100-4x25_seed0_native_v1",
    "backbone_revision": "9e250ed8f88b436472d2b24af26f82a8aa8c719d",
    "subset_dir": "/home/sakana/Code/AutoCheckout-CL/runs/modeling/training_runtime/ac818116-ceb0-4ce8-9c1e-979c6f54788a/subset",
    "task_config": "/home/sakana/Code/AutoCheckout-CL/data/processed/rpc_100-4x25_seed0_native_v1/task_config.json",
    "detector_dir": "/home/sakana/Code